# Inventory Allocation with Deep Q-Network (DQN)

This notebook trains and evaluates a DQN agent for inventory allocation across item-client pairs.

---

### Notebook Structure
| Cell | Description |
|------|-------------|
| 1 | Imports & Global Setup |
| 2 | `DataPreprocessor` Class |
| 3 | `DemandModel` Class |
| 4 | `InventoryEnvironment` Class |
| 5 | `DQN` & `ReplayBuffer`& `DQNAgent` Classes |
| 6 | Baseline Agent Classes |
| 7 | Helper Functions (`evaluate_baseline`, `train_agent`, `evaluate_agent`) |
| 8 | ⚙️ Configuration |
| 9 | Step 1–3: Data Preprocessing & Demand Modeling |
| 10 | Step 4: Environment & Agent Creation |
| 11 | Step 5: Training |
| 12 | Step 6: DQN Evaluation |
| 13 | Step 7: Baseline Comparisons |
| 14 | Step 8: Save All Results |

---
## Cell 1 — `Imports & Global Setup`

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List, Optional
import warnings
import pickle

warnings.filterwarnings('ignore')

device = torch.device("cpu")
print(f"Using device: CPU")

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

---
## Cell 2 — `DataPreprocessor` Class

In [ ]:
class DataPreprocessor:
    """Handles loading and preprocessing of transaction data"""
    def __init__(self, sales_files: List[str], unit_file: str):
        self.sales_files = sales_files
        self.unit_file = unit_file
        self.unit_conversion = None
        self.sales_data = None

    def load_unit_conversion(self) -> pd.DataFrame:
        print("Loading unit conversion data...")
        self.unit_conversion = pd.read_csv(self.unit_file)
        self.unit_conversion['η'] = (
            self.unit_conversion['ConvFact2'] / self.unit_conversion['ConvFact1']
        )
        return self.unit_conversion

    def load_sales_data(self) -> pd.DataFrame:
        print("Loading sales data...")
        all_sales = []
        for file in self.sales_files:
            try:
                df = pd.read_csv(file)
                all_sales.append(df)
                print(f"  Loaded {file}: {len(df)} records")
            except FileNotFoundError:
                print(f"  Warning: {file} not found, skipping...")
        if not all_sales:
            raise ValueError("No sales files were loaded successfully!")
        self.sales_data = pd.concat(all_sales, ignore_index=True)
        print(f"Total records loaded: {len(self.sales_data)}")
        return self.sales_data

    def normalize_quantities(self, sales_data: pd.DataFrame) -> pd.DataFrame:
        print("Normalizing quantities to main units...")
        merged = sales_data.merge(
            self.unit_conversion[['ItemId', 'Code', 'η']],
            left_on=['ItemId', 'ItemUnitCode'],
            right_on=['ItemId', 'Code'],
            how='left'
        )
        merged['NormalizedQuantity'] = merged['Quantity'] * merged['η']
        initial_count = len(merged)
        merged = merged.dropna(subset=['NormalizedQuantity'])
        removed = initial_count - len(merged)
        if removed > 0:
            print(f"  Removed {removed} records with undefined conversion factors")
        merged = merged[(merged['NormalizedQuantity'] > 0) & (merged['Price'] > 0)]
        print(f"  Valid records after normalization: {len(merged)}")
        return merged

    def prepare_data(self) -> pd.DataFrame:
        self.load_unit_conversion()
        sales = self.load_sales_data()
        sales = self.normalize_quantities(sales)
        sales['ProcessDate'] = pd.to_datetime(sales['ProcessDate'])
        sales = sales.sort_values('ProcessDate').reset_index(drop=True)
        sales['Revenue'] = sales['Price']
        print(f"\nData preparation complete!")
        print(f"  Date range: {sales['ProcessDate'].min()} to {sales['ProcessDate'].max()}")
        print(f"  Total clients: {sales['ClientId'].nunique()}")
        print(f"  Total items: {sales['ItemId'].nunique()}")
        print(f"  Total revenue: ${sales['Revenue'].sum():,.2f}")
        return sales

---
## Cell 3 — `DemandModel Class`

In [ ]:
class DemandModel:
    """Implements demand modeling per methodology"""
    def __init__(self, sales_data: pd.DataFrame):
        self.sales_data = sales_data
        self.daily_demand = None
        self.avg_demand = None
        self.gamma_params = {}
        self.ordering_probs = {}
        self.dates = None
        self.all_item_client_pairs = None

    def aggregate_daily_demand(self) -> pd.DataFrame:
        print("\nAggregating daily demand...")
        self.daily_demand = (
            self.sales_data
            .groupby(['ItemId', 'ClientId', 'ProcessDate'])
            .agg({'NormalizedQuantity': 'sum'})
            .reset_index()
            .rename(columns={'NormalizedQuantity': 'DailyDemand'})
        )
        self.dates = sorted(self.sales_data['ProcessDate'].unique())
        self.all_item_client_pairs = self.sales_data[['ItemId', 'ClientId']].drop_duplicates()
        print(f"  Unique dates: {len(self.dates)}")
        print(f"  Unique item-client pairs: {len(self.all_item_client_pairs)}")
        print(f"  Daily demand records (non-zero): {len(self.daily_demand)}")

        pairs_df = self.all_item_client_pairs.copy()
        pairs_df['_key'] = 1
        dates_df = pd.DataFrame({'ProcessDate': self.dates, '_key': 1})
        complete_demand = pairs_df.merge(dates_df, on='_key').drop('_key', axis=1)
        complete_demand = complete_demand.merge(
            self.daily_demand,
            on=['ItemId', 'ClientId', 'ProcessDate'],
            how='left'
        )
        complete_demand['DailyDemand'] = complete_demand['DailyDemand'].fillna(0.0)
        self.daily_demand = complete_demand
        print(f"  Complete demand grid size: {len(self.daily_demand)}")
        return self.daily_demand

    def compute_average_demand(self) -> pd.DataFrame:
        print("Computing average demand...")
        positive_demand = self.daily_demand[self.daily_demand['DailyDemand'] > 0]
        grouped = positive_demand.groupby(['ItemId', 'ClientId'])['DailyDemand']
        avg = grouped.mean()
        cnt = grouped.count()
        var = grouped.var(ddof=0)
        self.avg_demand = pd.DataFrame({
            'AvgDemand': avg,
            'TransactionDays': cnt,
            'VarDemand': var
        }).reset_index()
        total_days = len(self.dates)
        self.avg_demand['OrderingProb'] = self.avg_demand['TransactionDays'] / total_days
        mean_var = self.avg_demand['VarDemand'].mean()
        self.avg_demand['VarDemand'] = self.avg_demand['VarDemand'].fillna(mean_var)
        self.avg_demand['VarDemand'] = self.avg_demand['VarDemand'].clip(lower=0.01)
        print(f"  Item-client pairs: {len(self.avg_demand)}")
        print(f"  Avg demand stats: min={self.avg_demand['AvgDemand'].min():.2f}, "
              f"max={self.avg_demand['AvgDemand'].max():.2f}, "
              f"mean={self.avg_demand['AvgDemand'].mean():.2f}")
        return self.avg_demand

    def fit_gamma_distributions(self) -> Dict:
        print("Fitting Gamma distributions...")
        for _, row in self.avg_demand.iterrows():
            item_id = int(row['ItemId'])
            client_id = int(row['ClientId'])
            mean = float(row['AvgDemand'])
            var = float(row['VarDemand'])
            prob = float(row['OrderingProb'])
            if mean > 0 and var > 0:
                alpha = (mean ** 2) / var
                beta = mean / var
                alpha = np.clip(alpha, 0.01, 100)
                beta = np.clip(beta, 0.01, 100)
            else:
                alpha = 1.0
                beta = 1.0 / max(mean, 0.01)
            self.gamma_params[(item_id, client_id)] = {
                'alpha': alpha,
                'beta': beta,
                'mean': mean,
                'var': var
            }
            self.ordering_probs[(item_id, client_id)] = prob
        print(f"  Fitted parameters for {len(self.gamma_params)} pairs")
        return self.gamma_params

    def simulate_demand(self, item_id: int, client_id: int) -> float:
        params = self.gamma_params.get((item_id, client_id))
        prob = self.ordering_probs.get((item_id, client_id), 0.0)
        if params is None:
            return 0.0
        if np.random.random() < prob:
            demand = float(np.random.gamma(params['alpha'], 1.0 / params['beta']))
            return max(0.0, demand)
        return 0.0

---
## Cell 4 — `InventoryEnvironment Class`

In [ ]:
class InventoryEnvironment:
    def __init__(self, demand_model: DemandModel, active_pairs: List[Tuple]):
        self.demand_model = demand_model
        self.active_pairs = active_pairs
        self.unique_items = sorted(list(set([p[0] for p in active_pairs])))
        self.unique_clients = sorted(list(set([p[1] for p in active_pairs])))
        self.current_backlog = {pair: 0.0 for pair in self.active_pairs}
        self.previous_backlog = {pair: 0.0 for pair in self.active_pairs}
        self.current_inventory = {}
        self.current_day = 0
        self._compute_inventory_capacities()
        self._compute_transaction_density()
        self.action_multipliers = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
        self.n_actions = len(self.action_multipliers)
        self.λ_f = 1.0
        self.λ_s = 10.0
        self.λ_e = 0.2
        self.λ_b = 8.0
        print(f"\nEnvironment initialized:")
        print(f"  Unique items: {len(self.unique_items)}")
        print(f"  Unique clients: {len(self.unique_clients)}")
        print(f"  Total item-client pairs: {len(self.active_pairs)}")
        print(f"  Actions per item-client: {self.n_actions}")
        print(f"  Action multipliers: {self.action_multipliers}")
        print(f"  Backlog penalty weight (λ_b): {self.λ_b}")

    def _compute_inventory_capacities(self):
        print("\nComputing inventory capacities C_i...")
        daily_item_demand = (
            self.demand_model.sales_data
            .groupby(['ItemId', 'ProcessDate'])['NormalizedQuantity']
            .sum()
            .reset_index()
            .rename(columns={'NormalizedQuantity': 'DailyItemDemand'})
        )
        self.capacities = {}
        for item in self.unique_items:
            item_data = daily_item_demand[daily_item_demand['ItemId'] == item]
            if len(item_data) > 0:
                self.capacities[item] = float(item_data['DailyItemDemand'].max()) * 1.5
            else:
                if len(daily_item_demand) > 0:
                    self.capacities[item] = float(daily_item_demand['DailyItemDemand'].max()) * 1.5
                else:
                    self.capacities[item] = 100.0
            self.capacities[item] = max(self.capacities[item], 1.0)
        print(f"  Computed capacities for {len(self.capacities)} items")
        print(f"  Capacity stats: min={min(self.capacities.values()):.1f}, "
              f"max={max(self.capacities.values()):.1f}, "
              f"mean={np.mean(list(self.capacities.values())):.1f}")

    def _compute_transaction_density(self):
        print("Computing transaction density factors phi_t...")
        daily_transactions = (
            self.demand_model.sales_data
            .groupby('ProcessDate')
            .size()
            .reset_index(name='TransactionCount')
        )
        max_transactions = daily_transactions['TransactionCount'].max()
        daily_transactions['DensityFactor'] = daily_transactions['TransactionCount'] / max_transactions
        self.density_factors = dict(zip(daily_transactions['ProcessDate'],
                                        daily_transactions['DensityFactor']))
        self.all_dates = sorted(self.density_factors.keys())
        print(f"  Density factors computed for {len(self.all_dates)} days")
        print(f"  Max transactions: {max_transactions}")

    def reset(self):
        self.current_day = 0
        self.current_inventory = {item: float(self.capacities[item]) * 0.7
                                   for item in self.unique_items}
        self.current_backlog = {pair: 0.0 for pair in self.active_pairs}
        self.previous_backlog = {pair: 0.0 for pair in self.active_pairs}
        return self._get_state()

    def _get_state(self) -> Dict[Tuple, np.ndarray]:
        current_date = self.all_dates[min(self.current_day, len(self.all_dates) - 1)]
        phi_t = float(self.density_factors.get(current_date, 1.0))
        states = {}
        for item, client in self.active_pairs:
            I_it = float(self.current_inventory.get(item, 0.0))
            C_i = float(self.capacities[item])
            B_ict = float(self.current_backlog.get((item, client), 0.0))
            B_prev = float(self.previous_backlog.get((item, client), 0.0))
            params = self.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            Dbar = max(Dbar, 0.1)
            I_norm = I_it / max(C_i, 1.0)
            B_norm = B_ict / Dbar
            B_trend = (B_ict - B_prev) / Dbar
            D_norm = Dbar / max(C_i, 1.0)
            I_norm = np.clip(I_norm, 0.0, 2.0)
            B_norm = np.clip(B_norm, 0.0, 5.0)
            B_trend = np.clip(B_trend, -2.0, 2.0)
            D_norm = np.clip(D_norm, 0.0, 2.0)
            states[(item, client)] = np.array([
                I_norm, B_norm, B_trend, D_norm, phi_t
            ], dtype=np.float32)
        return states

    def _scale_allocations(self, desired_alloc: Dict[Tuple, float]) -> Dict[Tuple, float]:
        scaled = {}
        for item in self.unique_items:
            I_it = float(self.current_inventory.get(item, 0.0))
            desired_sum = sum(float(desired_alloc.get((item, client), 0.0))
                              for client in [c for i, c in self.active_pairs if i == item])
            scale = I_it / desired_sum if (desired_sum > I_it and desired_sum > 0) else 1.0
            for client in [c for i, c in self.active_pairs if i == item]:
                scaled[(item, client)] = float(desired_alloc.get((item, client), 0.0)) * scale
        return scaled

    def step(self, desired_allocations: Dict[Tuple, float]):
        for k in self.active_pairs:
            self.previous_backlog[k] = self.current_backlog[k]
        current_date = self.all_dates[min(self.current_day, len(self.all_dates) - 1)]
        phi_t = float(self.density_factors.get(current_date, 1.0))
        allocations = self._scale_allocations(desired_allocations)
        demands = {}
        for item, client in self.active_pairs:
            demands[(item, client)] = self.demand_model.simulate_demand(item, client)
        pair_rewards = {}
        for item, client in self.active_pairs:
            x = float(allocations.get((item, client), 0.0))
            D = float(demands.get((item, client), 0.0))
            B_prev = float(self.previous_backlog.get((item, client), 0.0))
            params = self.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            if Dbar < 0.1:
                Dbar = 0.1
            current_fulfill = min(x, D)
            r_current = self.λ_f * (current_fulfill / max(Dbar, 0.1))
            backlog_fulfill = max(0, min(x - D, B_prev))
            r_backlog_clear = self.λ_b * (backlog_fulfill / max(Dbar, 0.1))
            B_next = max(0.0, B_prev + D - x)
            self.current_backlog[(item, client)] = B_next
            service_level = 1.0 - min(1.0, B_next / max(Dbar, 0.1))
            r_service = self.λ_s * service_level
            pair_rewards[(item, client)] = r_current + r_backlog_clear + r_service
        total_reward = sum(pair_rewards.values())
        total_demand_per_item = {}
        for item in self.unique_items:
            total_demand_per_item[item] = sum(
                demands.get((item, c), 0.0)
                for c in [cl for i, cl in self.active_pairs if i == item]
            )
        next_inventory = {}
        for item in self.unique_items:
            C_i = float(self.capacities[item])
            total_D = total_demand_per_item.get(item, 0.0)
            if total_D < C_i * 0.3:
                next_inventory[item] = min(C_i * 0.9, C_i)
            elif total_D > C_i * 0.7:
                next_inventory[item] = C_i
            else:
                next_inventory[item] = min(max(C_i * 0.8, total_D * 2.0), C_i)
        self.current_day += 1
        if self.current_day < len(self.all_dates):
            self.current_inventory = next_inventory
        done = self.current_day >= len(self.all_dates)
        next_state = self._get_state()
        scaled_reward = total_reward / len(self.active_pairs)
        info = {
            'phi_t': phi_t,
            'demands': demands,
            'allocations': allocations,
            'backlog': self.current_backlog.copy(),
            'next_inventory': next_inventory,
            'pair_rewards': pair_rewards
        }
        return next_state, scaled_reward, done, info

---
## Cell 5 — `DQN Network, ReplayBuffer, and DQNAgent Classes`

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim: int, n_actions: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class ReplayBuffer:
    def __init__(self, capacity: int = 50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, transition: Tuple):
        self.buffer.append(transition)

    def sample(self, batch_size: int) -> List:
        return random.sample(self.buffer, batch_size)

    def __len__(self) -> int:
        return len(self.buffer)


class DQNAgent:
    def __init__(self, env: InventoryEnvironment, state_dim: int = 5,
                 learning_rate: float = 5e-4, gamma: float = 0.99,
                 epsilon_start: float = 1.0, epsilon_end: float = 0.05,
                 epsilon_decay: float = 0.995, tau: float = 0.005,
                 reg_lambda: float = 0.001):
        self.env = env
        self.state_dim = state_dim
        self.n_actions = env.n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.tau = tau
        self.reg_lambda = reg_lambda
        self.q_network = DQN(state_dim, self.n_actions)
        self.target_network = DQN(state_dim, self.n_actions)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        self.replay_buffer = ReplayBuffer(capacity=200000)

    def select_actions_batched(self, state_dict: Dict[Tuple[int, int], np.ndarray], epsilon: float = None):
        if epsilon is None:
            epsilon = self.epsilon
        keys = list(state_dict.keys())
        states_np = np.stack([state_dict[k] for k in keys], axis=0)
        states_t = torch.FloatTensor(states_np)
        N = states_t.shape[0]
        action_idx_np = np.empty(N, dtype=np.int64)
        rand_mask = (np.random.rand(N) < epsilon)
        if rand_mask.any():
            n_rand = rand_mask.sum()
            uniform_count = int(n_rand * 0.7)
            uniform_indices = np.random.choice(np.where(rand_mask)[0], uniform_count, replace=False)
            action_idx_np[uniform_indices] = np.random.randint(0, self.n_actions, size=uniform_count)
            high_indices = np.where(rand_mask)[0][~np.isin(np.where(rand_mask)[0], uniform_indices)]
            if len(high_indices) > 0:
                high_actions = [4, 5]
                action_idx_np[high_indices] = np.random.choice(high_actions, size=len(high_indices))
        if (~rand_mask).any():
            with torch.no_grad():
                q_vals = self.q_network(states_t)
                greedy = q_vals.argmax(dim=1).detach().cpu().numpy()
            action_idx_np[~rand_mask] = greedy[~rand_mask]
        action_indices = {}
        desired_allocations = {}
        action_list = []
        for idx, (item, client) in enumerate(keys):
            a = int(action_idx_np[idx])
            mult = self.env.action_multipliers[a]
            params = self.env.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            alloc = float(mult * Dbar)
            action_indices[(item, client)] = a
            desired_allocations[(item, client)] = alloc
            action_list.append(a)
        return action_indices, desired_allocations, action_list

    def train_step(self, batch_size: int = 32) -> float:
        if len(self.replay_buffer) < batch_size:
            return 0.0
        batch = self.replay_buffer.sample(batch_size)
        states, actions, rewards, next_states, dones, phi_ts, prev_allocs, curr_allocs = zip(*batch)
        states_t = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions).unsqueeze(1)
        rewards_t = torch.FloatTensor(rewards)
        next_states_t = torch.FloatTensor(np.array(next_states))
        dones_t = torch.FloatTensor(dones)
        phi_t = torch.FloatTensor(phi_ts)
        prev_alloc_t = torch.FloatTensor(prev_allocs)
        curr_alloc_t = torch.FloatTensor(curr_allocs)
        q_sa = self.q_network(states_t).gather(1, actions_t).squeeze(1)
        with torch.no_grad():
            next_q_max = self.target_network(next_states_t).max(1)[0]
        delta_x = torch.abs(curr_alloc_t - prev_alloc_t)
        y = rewards_t - self.reg_lambda * delta_x + self.gamma * next_q_max * (1.0 - dones_t)
        loss = (phi_t * (y - q_sa) ** 2).mean()
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()
        return float(loss.item())

    def soft_update_target(self):
        for t_p, p in zip(self.target_network.parameters(), self.q_network.parameters()):
            t_p.data.copy_(self.tau * p.data + (1 - self.tau) * t_p.data)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

---
## Cell 6 — `Baseline Agent Classes`

In [ ]:
class GreedyAgent:
    """Baseline 1: Always allocate exactly mean demand (1.0×)"""
    def __init__(self, env):
        self.env = env
        self.name = "Greedy (1.0×)"

    def select_actions(self, state_dict):
        allocations = {}
        for item, client in self.env.active_pairs:
            params = self.env.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            allocations[(item, client)] = Dbar
        return allocations


class RandomAgent:
    """Baseline 2: Random uniform action selection"""
    def __init__(self, env):
        self.env = env
        self.name = "Random Policy"

    def select_actions(self, state_dict):
        allocations = {}
        for item, client in self.env.active_pairs:
            params = self.env.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            action = np.random.randint(0, len(self.env.action_multipliers))
            mult = self.env.action_multipliers[action]
            allocations[(item, client)] = mult * Dbar
        return allocations


class ConservativeAgent:
    """Baseline 3: Always allocate conservatively (0.5×)"""
    def __init__(self, env):
        self.env = env
        self.name = "Conservative (0.5×)"

    def select_actions(self, state_dict):
        allocations = {}
        for item, client in self.env.active_pairs:
            params = self.env.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            allocations[(item, client)] = 0.5 * Dbar
        return allocations


class AggressiveAgent:
    """Baseline 4: Always allocate aggressively (3.0×)"""
    def __init__(self, env):
        self.env = env
        self.name = "Aggressive (3.0×)"

    def select_actions(self, state_dict):
        allocations = {}
        for item, client in self.env.active_pairs:
            params = self.env.demand_model.gamma_params.get((item, client), {'mean': 1.0})
            Dbar = float(params['mean'])
            allocations[(item, client)] = 3.0 * Dbar
        return allocations

---
## Cell 7 — `Helper Functions: evaluate_baseline, train_agent, evaluate_agent`

In [ ]:
def evaluate_baseline(env, agent, num_episodes=50):
    """Evaluate any baseline agent"""
    print(f"  Evaluating {agent.name}...")
    ep_rewards = []
    fulfillment_rates = []
    service_levels = []
    avg_backlogs = []
    all_keys = env.active_pairs
    for ep in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0.0
        total_demand = 0.0
        total_fulfilled = 0.0
        steps = 0
        backlog_tracker = {k: 0.0 for k in all_keys}
        while not done:
            allocations = agent.select_actions(state)
            next_state, r, done, info = env.step(allocations)
            demands = info['demands']
            allocs = info['allocations']
            for k in all_keys:
                D = float(demands.get(k, 0.0))
                x = float(allocs.get(k, 0.0))
                available = backlog_tracker[k] + D
                fulfill_qty = min(x, available)
                backlog_tracker[k] = available - fulfill_qty
                total_fulfilled += fulfill_qty
                total_demand += D
            total_reward += float(r)
            steps += 1
            state = next_state
        avg_r = total_reward / max(steps, 1)
        ep_rewards.append(avg_r)
        fulfill_rate = min(1.0, total_fulfilled / max(total_demand, 1e-6))
        fulfillment_rates.append(fulfill_rate)
        service_levels_per_pair = []
        final_backlog_values = []
        for k in all_keys:
            i, c = k
            params = env.demand_model.gamma_params.get((i, c), {'mean': 1.0})
            Dbar = float(params['mean'])
            final_backlog = backlog_tracker[k]
            final_backlog_values.append(final_backlog)
            if Dbar > 0:
                service = max(0.0, 1.0 - final_backlog / max(Dbar, 0.1))
            else:
                service = 1.0
            service_levels_per_pair.append(service)
        avg_service = np.mean(service_levels_per_pair)
        service_levels.append(avg_service)
        avg_backlogs.append(np.mean(final_backlog_values))
    return {
        'rewards': ep_rewards,
        'fulfillment_rate': fulfillment_rates,
        'service_level': service_levels,
        'avg_backlog': avg_backlogs,
        'mean_reward': np.mean(ep_rewards),
        'std_reward': np.std(ep_rewards),
        'mean_fulfillment': np.mean(fulfillment_rates),
        'mean_service': np.mean(service_levels),
        'mean_backlog': np.mean(avg_backlogs)
    }


def train_agent(env: InventoryEnvironment, agent: DQNAgent,
                num_episodes: int = 250, batch_size: int = 32,
                target_update_freq: int = 5):
    """Train the DQN agent"""
    rewards_hist, losses_hist = [], []
    print("\n" + "="*70)
    print("TRAINING DQN - WITH BACKLOG PENALTY AND BACKLOG TREND")
    print("="*70)
    all_keys = env.active_pairs
    for ep in range(num_episodes):
        state = env.reset()
        done = False
        prev_alloc = {k: 0.0 for k in all_keys}
        ep_reward, ep_loss, steps = 0.0, 0.0, 0
        while not done:
            action_idx, desired_alloc, _ = agent.select_actions_batched(state, epsilon=agent.epsilon)
            next_state, r, done, info = env.step(desired_alloc)
            phi_t = info['phi_t']
            scaled_alloc = info['allocations']
            for k in all_keys:
                s_vec = state[k]
                ns_vec = next_state[k]
                a = int(action_idx[k])
                curr_alloc = float(scaled_alloc.get(k, 0.0))
                transition = (
                    s_vec, a, float(r), ns_vec, float(done),
                    float(phi_t), float(prev_alloc[k]), float(curr_alloc),
                )
                agent.replay_buffer.push(transition)
                prev_alloc[k] = curr_alloc
            loss = agent.train_step(batch_size=batch_size)
            ep_reward += float(r)
            ep_loss += float(loss)
            steps += 1
            state = next_state
        avg_r = ep_reward / max(steps, 1)
        avg_l = ep_loss / max(steps, 1)
        rewards_hist.append(avg_r)
        losses_hist.append(avg_l)
        if ep % target_update_freq == 0:
            agent.soft_update_target()
        agent.decay_epsilon()
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"Episode {ep+1:3d}/{num_episodes} | "
                  f"Avg Reward: {avg_r:8.4f} | "
                  f"Avg Loss: {avg_l:8.4f} | "
                  f"ε: {agent.epsilon:.3f} | "
                  f"Steps: {steps}")
    return rewards_hist, losses_hist


def evaluate_agent(agent: DQNAgent, env: InventoryEnvironment, num_episodes: int = 50):
    """Evaluate the trained agent and collect action history"""
    print("\n" + "="*70)
    print("EVALUATION - WITH BACKLOG PENALTY AND BACKLOG TREND")
    print("="*70)
    agent.q_network.eval()
    ep_rewards = []
    fulfillment_rates = []
    service_levels = []
    avg_backlogs = []
    all_actions = []
    all_keys = env.active_pairs
    for ep in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0.0
        total_demand = 0.0
        total_fulfilled = 0.0
        steps = 0
        backlog_tracker = {k: 0.0 for k in all_keys}
        while not done:
            action_idx, desired_alloc, action_list = agent.select_actions_batched(state, epsilon=0.0)
            all_actions.extend(action_list)
            next_state, r, done, info = env.step(desired_alloc)
            demands = info['demands']
            allocs = info['allocations']
            for k in all_keys:
                D = float(demands.get(k, 0.0))
                x = float(allocs.get(k, 0.0))
                available = backlog_tracker[k] + D
                fulfill_qty = min(x, available)
                backlog_tracker[k] = available - fulfill_qty
                total_fulfilled += fulfill_qty
                total_demand += D
            total_reward += float(r)
            steps += 1
            state = next_state
        avg_r = total_reward / max(steps, 1)
        ep_rewards.append(avg_r)
        fulfill_rate = min(1.0, total_fulfilled / max(total_demand, 1e-6))
        fulfillment_rates.append(fulfill_rate)
        service_levels_per_pair = []
        final_backlog_values = []
        for k in all_keys:
            i, c = k
            params = env.demand_model.gamma_params.get((i, c), {'mean': 1.0})
            Dbar = float(params['mean'])
            final_backlog = backlog_tracker[k]
            final_backlog_values.append(final_backlog)
            if Dbar > 0:
                service = max(0.0, 1.0 - final_backlog / max(Dbar, 0.1))
            else:
                service = 1.0
            service_levels_per_pair.append(service)
        avg_service = np.mean(service_levels_per_pair)
        service_levels.append(avg_service)
        avg_backlogs.append(np.mean(final_backlog_values))
        if (ep + 1) % 10 == 0:
            print(f"Eval ep {ep+1:2d}/{num_episodes} | "
                  f"Avg reward: {avg_r:.4f} | "
                  f"Fulfillment: {fulfill_rate:.2%} | "
                  f"Service level: {avg_service:.2%}")
    agent.q_network.train()
    metrics = {
        "fulfillment_rate": fulfillment_rates,
        "service_level": service_levels,
        "avg_backlog": avg_backlogs,
        "action_history": all_actions
    }
    return ep_rewards, metrics

---
## Cell 8 — `Configuration`

In [ ]:
print("\n" + "="*70)
print(" STARTING TRAINING FOR INVENTORY ALLOCATION DQN")
print("="*70)

# ============== CONFIGURATION ==============
N_TOP_PAIRS     = 3000
N_EPISODES      = 350
N_EVAL_EPISODES = 50
BATCH_SIZE      = 128

sales_files = [
    "Sale_012025.csv", "Sale_022025.csv", "Sale_032025.csv",
    "Sale_042025.csv", "Sale_052025.csv", "Sale_062025.csv",
    "Sale_072025.csv",
]
unit_file = "Unit.csv"

# Create results directory
os.makedirs("./training_results", exist_ok=True)

---
## Cell 9 — `Steps 1–3: Data Preprocessing & Demand Modeling`

In [ ]:
print("\n" + "="*70)
print("STEP 1-3: DATA PREPROCESSING & DEMAND MODELING")
print("="*70)

preprocessor = DataPreprocessor(sales_files, unit_file)
sales_data = preprocessor.prepare_data()

# Select top N item-client pairs by total revenue
pair_revenue = sales_data.groupby(['ItemId', 'ClientId'])['Revenue'].sum().sort_values(ascending=False)
top_pairs = pair_revenue.head(N_TOP_PAIRS).index.tolist()

# Filter sales data to top pairs only
filtered_sales = sales_data[sales_data.set_index(['ItemId', 'ClientId']).index.isin(top_pairs)].copy()

# Build demand model on filtered data
demand_model = DemandModel(filtered_sales)
demand_model.aggregate_daily_demand()
demand_model.compute_average_demand()
demand_model.fit_gamma_distributions()

---
## Cell 10 — `Step 4: Environment & Agent Creation`

In [ ]:
print("\n" + "="*70)
print("STEP 4: ENVIRONMENT & AGENT CREATION")
print("="*70)

my_env = InventoryEnvironment(demand_model, top_pairs)

dqn_agent = DQNAgent(
    my_env,
    state_dim=5,
    learning_rate=5e-4,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    tau=0.005,
    reg_lambda=0.001,
)

---
## Cell 11 — `Step 5: Training`

In [ ]:
print("\n" + "="*70)
print(f"STEP 5: TRAINING - {my_env.n_actions} ACTIONS, {N_EPISODES} EPISODES")
print("="*70)

train_rewards, train_losses = train_agent(
    my_env, dqn_agent,
    num_episodes=N_EPISODES,
    batch_size=BATCH_SIZE,
    target_update_freq=10
)

---
## Cell 12 — `Step 6: Evaluate DQN Agent`

In [ ]:
print("\n" + "="*70)
print("STEP 6: EVALUATE DQN AGENT")
print("="*70)

eval_rewards, eval_metrics = evaluate_agent(
    dqn_agent, my_env,
    num_episodes=N_EVAL_EPISODES
)

---
## Cell 13 — `Step 7: Baseline Comparisons`

In [ ]:
print("\n" + "="*70)
print("STEP 7: BASELINE COMPARISONS")
print("="*70)

# Instantiate baseline agents
greedy_agent      = GreedyAgent(my_env)
random_agent      = RandomAgent(my_env)
conservative_agent = ConservativeAgent(my_env)
aggressive_agent  = AggressiveAgent(my_env)

# Evaluate all baselines
print("\nEvaluating baseline policies...")
baseline_results = {}

baseline_results['greedy']       = evaluate_baseline(my_env, greedy_agent,       num_episodes=N_EVAL_EPISODES)
baseline_results['random']       = evaluate_baseline(my_env, random_agent,       num_episodes=N_EVAL_EPISODES)
baseline_results['conservative'] = evaluate_baseline(my_env, conservative_agent, num_episodes=N_EVAL_EPISODES)
baseline_results['aggressive']   = evaluate_baseline(my_env, aggressive_agent,   num_episodes=N_EVAL_EPISODES)

# Print comparison table
print("\n" + "-"*80)
print("BASELINE COMPARISON RESULTS:")
print("-"*80)
print(f"{'Policy':<20} {'Reward':<20} {'Fulfillment':<15} {'Service':<15} {'Backlog':<10}")
print("-"*80)

print(f"{'DQN (Ours)':<20} {np.mean(eval_rewards):.3f} ± {np.std(eval_rewards):.3f}   "
      f"{np.mean(eval_metrics['fulfillment_rate']):.1%}   "
      f"{np.mean(eval_metrics['service_level']):.1%}   "
      f"{np.mean(eval_metrics['avg_backlog']):.2f}")

print(f"{greedy_agent.name:<20} {baseline_results['greedy']['mean_reward']:.3f} ± {baseline_results['greedy']['std_reward']:.3f}   "
      f"{baseline_results['greedy']['mean_fulfillment']:.1%}   "
      f"{baseline_results['greedy']['mean_service']:.1%}   "
      f"{baseline_results['greedy']['mean_backlog']:.2f}")

print(f"{random_agent.name:<20} {baseline_results['random']['mean_reward']:.3f} ± {baseline_results['random']['std_reward']:.3f}   "
      f"{baseline_results['random']['mean_fulfillment']:.1%}   "
      f"{baseline_results['random']['mean_service']:.1%}   "
      f"{baseline_results['random']['mean_backlog']:.2f}")

print(f"{conservative_agent.name:<20} {baseline_results['conservative']['mean_reward']:.3f} ± {baseline_results['conservative']['std_reward']:.3f}   "
      f"{baseline_results['conservative']['mean_fulfillment']:.1%}   "
      f"{baseline_results['conservative']['mean_service']:.1%}   "
      f"{baseline_results['conservative']['mean_backlog']:.2f}")

print(f"{aggressive_agent.name:<20} {baseline_results['aggressive']['mean_reward']:.3f} ± {baseline_results['aggressive']['std_reward']:.3f}   "
      f"{baseline_results['aggressive']['mean_fulfillment']:.1%}   "
      f"{baseline_results['aggressive']['mean_service']:.1%}   "
      f"{baseline_results['aggressive']['mean_backlog']:.2f}")

print("-"*80)

# Save baseline results
with open("./training_results/baseline_results.pkl", "wb") as f:
    pickle.dump(baseline_results, f)

print("\n Baseline comparisons complete! Saved to ./training_results/baseline_results.pkl")

---
## Cell 14 — `Save All Results to Disk`

In [ ]:
print("\n" + "="*70)
print("SAVING ALL RESULTS TO DISK")
print("="*70)

# Save model weights
torch.save(dqn_agent.q_network.state_dict(), "./training_results/trained_model.pth")

# Save training history
np.save("./training_results/train_rewards.npy", train_rewards)
np.save("./training_results/train_losses.npy",  train_losses)

# Save evaluation results
np.save("./training_results/eval_rewards.npy",      eval_rewards)
np.save("./training_results/eval_fulfillment.npy",  eval_metrics['fulfillment_rate'])
np.save("./training_results/eval_service.npy",      eval_metrics['service_level'])
np.save("./training_results/eval_backlog.npy",      eval_metrics['avg_backlog'])
np.save("./training_results/action_history.npy",    eval_metrics['action_history'])

# Save run config & summary stats
config = {
    'n_pairs':             len(my_env.active_pairs),
    'n_actions':           my_env.n_actions,
    'action_multipliers':  my_env.action_multipliers,
    'n_episodes':          N_EPISODES,
    'n_eval_episodes':     N_EVAL_EPISODES,
    'batch_size':          BATCH_SIZE,
    'train_rewards_mean':  np.mean(train_rewards[-50:]),
    'train_rewards_std':   np.std(train_rewards[-50:]),
    'eval_rewards_mean':   np.mean(eval_rewards),
    'eval_rewards_std':    np.std(eval_rewards),
    'fulfillment_mean':    np.mean(eval_metrics['fulfillment_rate']),
    'fulfillment_std':     np.std(eval_metrics['fulfillment_rate']),
    'service_mean':        np.mean(eval_metrics['service_level']),
    'service_std':         np.std(eval_metrics['service_level']),
    'backlog_mean':        np.mean(eval_metrics['avg_backlog']),
    'backlog_std':         np.std(eval_metrics['avg_backlog']),
    'lambda_b':            my_env.λ_b,
    'lambda_s':            my_env.λ_s,
    'lambda_f':            my_env.λ_f,
}

with open("./training_results/config.pkl", "wb") as f:
    pickle.dump(config, f)

print("\n ALL TRAINING COMPLETE! Results saved to ./training_results/")
print(f"    Model:           ./training_results/trained_model.pth")
print(f"    Training curves: ./training_results/train_rewards.npy")
print(f"    Action history:  ./training_results/action_history.npy")
print(f"    Baseline results:./training_results/baseline_results.pkl")
print("\n" + "="*70)
print(" NOW RUN CELL 2 TO GENERATE PUBLICATION-QUALITY PLOTS")
print("="*70)